In [1]:
from scipy.stats import chi2, norm
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.families.links import Log, CLogLog, Logit
from statsmodels.genmod.families.links import Identity
from scipy.special import logit, expit
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

pd.set_option("display.width", 2000)

# import custom functions
from ST5213_helpers import *

In [2]:
df = pd.read_csv('dataset/snails.txt', sep=' ')
print(df.head())

  Species  Exposure  Rel_Hum  Temp  Deaths   N
1       A         1     60.0    10       0  20
2       A         1     60.0    15       0  20
3       A         1     60.0    20       0  20
4       A         1     65.8    10       0  20
5       A         1     65.8    15       0  20


In [3]:
print(df.info())
print(df['Exposure'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 1 to 96
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   96 non-null     str    
 1   Exposure  96 non-null     int64  
 2   Rel_Hum   96 non-null     float64
 3   Temp      96 non-null     int64  
 4   Deaths    96 non-null     int64  
 5   N         96 non-null     int64  
dtypes: float64(1), int64(4), str(1)
memory usage: 4.6 KB
None
Exposure
1    24
2    24
3    24
4    24
Name: count, dtype: int64


In [4]:
df['Species'] = pd.Categorical(df['Species'])
df['Exposure'] = pd.Categorical(df['Exposure'], categories=[4, 3, 2, 1], ordered=True)
df['Rel_Hum'] = pd.Categorical(df['Rel_Hum'])
df['Temp'] = pd.Categorical(df['Temp'])

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 1 to 96
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Species   96 non-null     category
 1   Exposure  96 non-null     category
 2   Rel_Hum   96 non-null     category
 3   Temp      96 non-null     category
 4   Deaths    96 non-null     int64   
 5   N         96 non-null     int64   
dtypes: category(4), int64(2)
memory usage: 2.3 KB
None


In [5]:
df.pivot_table(index='Species', columns='Exposure', values='Deaths', aggfunc='sum')

Exposure,4,3,2,1
Species,,,,
A,55,24,3,0
B,113,69,11,0


In [6]:
df.pivot_table(index='Species', columns='Rel_Hum', values='Deaths', aggfunc='sum')

Rel_Hum,60.0,65.8,70.5,75.8
Species,,,,
A,33,22,16,11
B,76,55,34,28


In [7]:
df.pivot_table(index='Species', columns='Temp', values='Deaths', aggfunc='sum')

Temp,10,15,20
Species,,,
A,17,28,37
B,47,66,80


In [8]:
df['Survive'] = df['N'] - df['Deaths']

fm = glm('Survive + Deaths ~ (Species + Exposure + Rel_Hum + Temp)**4', df, family=Binomial()).fit() 

res = stepAIC(fm, verbose=True)

c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linea

Start AIC: 350.862599
Start formula: Survive + Deaths ~ (Species + Exposure + Rel_Hum + Temp)**4
Best drop: Species:Exposure:Rel_Hum:Temp -> AIC 317.451390 (improve 33.4112)
Best drop: Exposure:Rel_Hum:Temp -> AIC 289.592050 (improve 27.8593)
Best drop: Species:Exposure:Rel_Hum -> AIC 274.322339 (improve 15.2697)
Best drop: Exposure:Rel_Hum -> AIC 257.887632 (improve 16.4347)
Best drop: Species:Rel_Hum:Temp -> AIC 246.943326 (improve 10.9443)
Best drop: Rel_Hum:Temp -> AIC 236.182452 (improve 10.7609)
Best drop: Species:Exposure:Temp -> AIC 228.911873 (improve 7.27058)
Best drop: Species:Exposure -> AIC 223.283511 (improve 5.62836)
Best drop: Exposure:Temp -> AIC 218.052501 (improve 5.23101)
Best drop: Species:Rel_Hum -> AIC 213.642921 (improve 4.40958)
Best drop: Species:Temp -> AIC 209.669114 (improve 3.97381)
Best drop: Temp -> AIC 230.482798 (improve -20.8137)


In [9]:
res.selected_formula

'Survive + Deaths ~ Species + Exposure + Rel_Hum + Temp'

In [10]:
drop1_glm(res.model)

,Df,Deviance,AIC,LRT,Pr(>Chi)
<none>,,30.806516,209.669114,,
Species,1,100.262384,277.124983,69.455868,0.0
Exposure,3,416.152565,589.015164,385.34605,0.0
Rel_Hum,3,95.289595,268.152194,64.48308,0.0
Temp,2,55.620199,230.482798,24.813684,0.000004


In [11]:
print(drop1_glm(fm))

                               Df      Deviance         AIC       LRT  Pr(>Chi)
<none>                             4.757297e-09  350.862599                    
Species:Exposure:Rel_Hum:Temp  18  2.588791e+00  317.451390  2.588791  0.999991


In [12]:
fm = update_glm(fm, drop='Species:Exposure:Rel_Hum:Temp')
print(drop1_glm(fm))

                          Df   Deviance         AIC       LRT  Pr(>Chi)
<none>                         2.588791  317.451390                    
Species:Exposure:Rel_Hum   9   4.291460  301.154059  1.702669  0.995346
Species:Exposure:Temp      6   7.268328  310.130927  4.679537  0.585521
Species:Rel_Hum:Temp       6   3.809641  306.672240  1.220849  0.975842
Exposure:Rel_Hum:Temp     18  10.729451  289.592050   8.14066  0.976468


In [13]:
fm = update_glm(fm, drop='Species:Exposure:Rel_Hum')
print(drop1_glm(fm))

                       Df   Deviance         AIC      LRT  Pr(>Chi)
<none>                      4.291460  301.154059                   
Species:Exposure:Temp   6   9.542260  294.404859   5.2508  0.512072
Species:Rel_Hum:Temp    6   5.331010  290.193608  1.03955  0.984069
Exposure:Rel_Hum:Temp  18  13.459741  274.322339  9.16828  0.955716


In [14]:
fm = update_glm(fm, drop='Species:Rel_Hum:Temp')
print(drop1_glm(fm))

                       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>                      5.331010  290.193608                    
Species:Rel_Hum         3   6.779668  285.642267  1.448659   0.69417
Species:Exposure:Temp   6  10.265017  283.127616  4.934007  0.552305
Exposure:Rel_Hum:Temp  18  14.559877  263.422476  9.228868  0.954203


In [15]:
fm = update_glm(fm, drop='Exposure:Rel_Hum:Temp')
print(drop1_glm(fm))

                      Df   Deviance         AIC       LRT  Pr(>Chi)
<none>                    14.559877  263.422476                    
Species:Rel_Hum        3  15.871463  258.734062  1.311586  0.726384
Exposure:Rel_Hum       9  16.080727  246.943326   1.52085  0.996989
Rel_Hum:Temp           6  15.860292  252.722890  1.300415  0.971635
Species:Exposure:Temp  6  19.355910  256.218508  4.796032  0.570227


In [16]:
fm = update_glm(fm, drop='Exposure:Rel_Hum')
print(drop1_glm(fm))

                      Df   Deviance         AIC       LRT  Pr(>Chi)
<none>                    16.080727  246.943326                    
Species:Rel_Hum        3  17.413432  242.276031  1.332705  0.721382
Rel_Hum:Temp           6  17.319853  236.182452  1.239126  0.974908
Species:Exposure:Temp  6  20.838147  239.700745   4.75742  0.575281


In [17]:
fm = update_glm(fm, drop='Rel_Hum:Temp')
print(drop1_glm(fm))

                      Df   Deviance         AIC       LRT  Pr(>Chi)
<none>                    17.319853  236.182452                    
Species:Rel_Hum        3  18.791549  231.654148  1.471696  0.688818
Species:Exposure:Temp  6  22.049275  228.911873  4.729421  0.578955


In [18]:
fm = update_glm(fm, drop='Species:Rel_Hum')
print(drop1_glm(fm))

                      Df   Deviance         AIC        LRT  Pr(>Chi)
<none>                    18.791549  231.654148                     
Rel_Hum                3  83.129867  289.992465  64.338318       0.0
Species:Exposure:Temp  6  23.582689  224.445287    4.79114  0.570867


In [19]:
fm = update_glm(fm, drop='Species:Exposure:Temp')
print(drop1_glm(fm))

                 Df   Deviance         AIC        LRT  Pr(>Chi)
<none>               23.582689  224.445287                     
Rel_Hum           3  87.978375  282.840974  64.395687       0.0
Species:Exposure  3  24.095522  218.958120   0.512833  0.916063
Species:Temp      2  23.847665  220.710263   0.264976  0.875913
Exposure:Temp     6  30.511718  219.374316   6.929029   0.32746


In [20]:
fm = update_glm(fm, drop='Species:Exposure')
print(drop1_glm(fm))

              Df   Deviance         AIC        LRT  Pr(>Chi)
<none>            24.095522  218.958120                     
Rel_Hum        3  88.630134  277.492732  64.534612       0.0
Species:Temp   2  24.265135  215.127734   0.169614   0.91869
Exposure:Temp  6  30.780323  213.642921   6.684801  0.350982


In [21]:
fm = update_glm(fm, drop='Species:Temp')
print(drop1_glm(fm))

              Df   Deviance         AIC        LRT  Pr(>Chi)
<none>            24.265135  215.127734                     
Species        1  93.742479  282.605078  69.477344       0.0
Rel_Hum        3  88.817122  273.679721  64.551987       0.0
Exposure:Temp  6  30.806516  209.669114   6.541381  0.365347


In [22]:
fm = update_glm(fm, drop='Exposure:Temp')
print(drop1_glm(fm))

         Df    Deviance         AIC        LRT  Pr(>Chi)
<none>        30.806516  209.669114                     
Species   1  100.262384  277.124983  69.455868       0.0
Exposure  3  416.152565  589.015164  385.34605       0.0
Rel_Hum   3   95.289595  268.152194   64.48308       0.0
Temp      2   55.620199  230.482798  24.813684  0.000004


In [23]:
print(chi2.sf(fm.deviance, fm.df_resid),'\n')

0.9999999939585751 



In [24]:
print(fm.summary2())

                   Results: Generalized linear model
Model:                GLM                     Method:           IRLS    
Model Family:         Binomial                AIC:              209.6691
Link Function:        Logit                   BIC:              235.3126
Dependent Variable:   ['Survive', 'Deaths']   Log-Likelihood:   -94.835 
Date:                 2026-10-06 14:59        LL-Null:          -349.29 
No. Observations:     96                      Deviance:         30.807  
Df Model:             9                       Pearson chi2:     26.5    
Df Residuals:         86                      Scale:            1.0000  
------------------------------------------------------------------------
                 Coef.   Std.Err.     z    P>|z|     [0.025     0.975]  
------------------------------------------------------------------------
Intercept        1.0307     0.2152  4.7891 0.0000      0.6089     1.4525
Species[T.B]    -1.2895     0.1618 -7.9694 0.0000     -1.6066    -0.972

In [25]:
np.exp(1.2895)

np.float64(3.630970616497579)

In [26]:
df2 = pd.read_csv('dataset/snails.txt', sep=' ')
df2['Species'] = pd.Categorical(df2['Species'])
df2['Survive'] = df2['N'] - df2['Deaths']

fm2 = glm('Survive + Deaths ~ Species/(Exposure + Rel_Hum + Temp)', df2, family=Binomial()).fit() 
print(fm2.summary2())

                 Results: Generalized linear model
Model:                GLM                   Method:         IRLS    
Model Family:         Binomial              AIC:            228.8516
Link Function:        Logit                 BIC:            249.3664
Dependent Variable:   ['Survive', 'Deaths'] Log-Likelihood: -106.43 
Date:                 2026-10-06 15:02      LL-Null:        -349.29 
No. Observations:     96                    Deviance:       53.989  
Df Model:             7                     Pearson chi2:   44.6    
Df Residuals:         88                    Scale:          1.0000  
--------------------------------------------------------------------
                     Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------
Intercept            2.5047   1.6467   1.5210 0.1283 -0.7229  5.7322
Species[T.B]        -3.0554   2.0393  -1.4983 0.1341 -7.0524  0.9416
Species[A]:Exposure -1.4632   0.1822  -8.0313 0.0000

In [27]:
2.5047-3.0554

-0.5507

In [28]:
fm3 = glm('Survive + Deaths ~ Species + Exposure + Rel_Hum + Temp', df2, family=Binomial()).fit() 
print(fm3.summary2())

                Results: Generalized linear model
Model:              GLM                   Method:         IRLS    
Model Family:       Binomial              AIC:            223.9323
Link Function:      Logit                 BIC:            236.7541
Dependent Variable: ['Survive', 'Deaths'] Log-Likelihood: -106.97 
Date:               2026-10-06 15:05      LL-Null:        -349.29 
No. Observations:   96                    Deviance:       55.070  
Df Model:           4                     Pearson chi2:   45.9    
Df Residuals:       91                    Scale:          1.0000  
-------------------------------------------------------------------
               Coef.   Std.Err.     z      P>|z|    [0.025   0.975]
-------------------------------------------------------------------
Intercept      1.4049    0.9707    1.4474  0.1478  -0.4976   3.3075
Species[T.B]  -1.3086    0.1635   -8.0040  0.0000  -1.6291  -0.9882
Exposure      -1.5034    0.1024  -14.6886  0.0000  -1.7040  -1.3028
Rel_Hu

In [29]:
1.4049-1.3086

0.09630000000000005

In [30]:
print(anova_nested(fm3, fm2))

         Resid. Df  Resid. Dev   Df  Deviance  Pr(>Chi)
Model 1       91.0   55.069750  NaN       NaN       NaN
Model 2       88.0   53.989001  3.0  1.080749  0.781723


In [31]:
fm4 = update_glm(fm3, add='I(Exposure**2)')
fm5 = update_glm(fm4, add='I(Exposure**3)')
print(anova_nested(fm3, fm4, fm5))

         Resid. Df  Resid. Dev   Df   Deviance  Pr(>Chi)
Model 1       91.0   55.069750  NaN        NaN       NaN
Model 2       90.0   32.902269  1.0  22.167482  0.000002
Model 3       89.0   32.150913  1.0   0.751355  0.386047


In [32]:
print(drop1_glm(fm4))

                 Df    Deviance         AIC        LRT  Pr(>Chi)
<none>                32.902269  203.764867                     
Species           1  102.295692  271.158291  69.393423       0.0
Exposure          1   79.798054  248.660652  46.895785       0.0
Rel_Hum           1   96.483670  265.346269  63.581402       0.0
Temp              1   57.305495  226.168093  24.403226  0.000001
I(Exposure ** 2)  1   55.069750  223.932349  22.167482  0.000002


In [33]:
print(fm4.summary2().tables[1].round(3))

                  Coef.  Std.Err.      z  P>|z|  [0.025  0.975]
Intercept         7.734     1.976  3.913    0.0   3.860  11.607
Species[T.B]     -1.288     0.162 -7.969    0.0  -1.605  -0.971
Exposure         -5.837     1.127 -5.181    0.0  -8.045  -3.629
Rel_Hum           0.105     0.014  7.656    0.0   0.078   0.132
Temp             -0.093     0.019 -4.859    0.0  -0.131  -0.056
I(Exposure ** 2)  0.702     0.175  4.015    0.0   0.359   1.044


In [37]:
df2.describe()

,Exposure,Rel_Hum,Temp,Deaths,N,Survive
count,96.000000,96.000000,96.000000,96.000000,96.0,96.000000
mean,2.500000,68.025000,15.000000,2.864583,20.0,17.135417
std,1.123903,5.859998,4.103913,3.763033,0.0,3.763033
min,1.000000,60.000000,10.000000,0.000000,20.0,4.000000
25%,1.750000,64.350000,10.000000,0.000000,20.0,15.750000
50%,2.500000,68.150000,15.000000,1.000000,20.0,19.000000
75%,3.250000,71.825000,20.000000,4.250000,20.0,20.000000
max,4.000000,75.800000,20.000000,16.000000,20.0,20.000000


In [36]:
np.exp(-fm4.params)

Intercept             0.000438
Species[T.B]          3.625584
Exposure            342.781191
Rel_Hum               0.899883
Temp                  1.097581
I(Exposure ** 2)      0.495729
dtype: float64

In [38]:
beta  = np.array(-fm4.params)
def f(x):

    return np.exp(beta[2]*x+beta[5]*x**2)

f(4)/f(1), f(4)/f(2), f(4)/f(3)

(np.float64(1080.7405203076166),
 np.float64(25.880362449933834),
 np.float64(2.521911697019747))